# 99: Run all and validate

**Why:** Run this notebook after setup and dataset submissions. It executes notebooks with `nbclient` when available, aggregates the registry, checks every CF file, and writes the Drive README index.

The orchestrator re-raises failures so Colab exits visibly non-zero.

In [ ]:
# Local bootstrap. Installs are quiet so the notebook stays readable.
%pip -q install earthengine-api xee xarray netCDF4 h5netcdf geopandas rioxarray regionmask "dask[diagnostics]" requests tqdm python-dotenv pyyaml matplotlib pandas pyproj shapely compliance-checker
from pathlib import Path
import os, sys, json, time, logging, traceback
from datetime import datetime

PROJECT_ID = os.getenv("GEE_PROJECT", "ee-ishansinhagzb")
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

# Local files hold configuration, logs, and task metadata. GeoTIFF outputs go to Google Drive via EE.
DRIVE_ROOT = Path(os.getenv("OGALLALA_PHASE0_ROOT", "./Ogallala_Phase0"))
SETUP_DIR = DRIVE_ROOT / '00_setup'
SETUP_DIR.mkdir(parents=True, exist_ok=True)
UTILS_SOURCE = '"""Shared utilities for the Ogallala Phase 0 Colab notebooks."""\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport math\nimport os\nimport time\nimport traceback\nfrom datetime import datetime, timezone\nfrom functools import wraps\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\nimport numpy as np\nimport pandas as pd\n\nREGISTRY_COLUMNS = [\n    "timestamp", "notebook_id", "dataset", "task_id_or_file", "pathway",\n    "status", "start_time", "end_time", "n_bytes", "error_message", "retries",\n]\nTERMINAL = {"COMPLETED", "FAILED"}\n\n\ndef utc_now() -> str:\n    """Return an ISO-8601 UTC timestamp."""\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef ensure_layout(root: Path, leaf: str) -> Path:\n    """Create the standard Drive tree and return one dataset leaf."""\n    root = Path(root)\n    for path in [root / "00_setup", root / "logs", root / "manifests"]:\n        path.mkdir(parents=True, exist_ok=True)\n    for name in ["raw", "cf", "qc"]:\n        (root / leaf / name).mkdir(parents=True, exist_ok=True)\n    registry = root / "logs" / "task_registry.csv"\n    if not registry.exists():\n        pd.DataFrame(columns=REGISTRY_COLUMNS).to_csv(registry, index=False)\n    return root / leaf\n\n\ndef configure_logger(notebook_id: str, root: Path) -> tuple[logging.Logger, Path]:\n    """Configure rotating file and stdout logging for a notebook."""\n    from logging.handlers import RotatingFileHandler\n    log_dir = Path(root) / "logs"\n    log_dir.mkdir(parents=True, exist_ok=True)\n    path = log_dir / f"{notebook_id}_{datetime.now().strftime(\'%Y%m%dT%H%M%S\')}.log"\n    logger = logging.getLogger(notebook_id)\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")\n    file_handler = RotatingFileHandler(path, maxBytes=50 * 1024 * 1024, backupCount=3)\n    file_handler.setFormatter(formatter)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n    logger.addHandler(stream_handler)\n    return logger, path\n\n\ndef registry_path(root: Path) -> Path:\n    """Return the shared task registry path."""\n    path = Path(root) / "logs" / "task_registry.csv"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        pd.DataFrame(columns=REGISTRY_COLUMNS).to_csv(path, index=False)\n    return path\n\n\ndef append_registry(root: Path, **values: Any) -> None:\n    """Append a normalized task or file event to the registry."""\n    row = {column: values.get(column, "") for column in REGISTRY_COLUMNS}\n    row["timestamp"] = row["timestamp"] or utc_now()\n    path = registry_path(root)\n    frame = pd.DataFrame([row], columns=REGISTRY_COLUMNS)\n    frame.to_csv(path, mode="a", header=path.stat().st_size == 0, index=False)\n\n\ndef logged_step(notebook_id: str, dataset: str, root: Path, logger: logging.Logger) -> Callable:\n    """Decorate an I/O step so failures are logged and recorded."""\n    def decorator(function: Callable) -> Callable:\n        @wraps(function)\n        def wrapper(*args: Any, **kwargs: Any) -> Any:\n            start = utc_now()\n            try:\n                result = function(*args, **kwargs)\n                append_registry(root, notebook_id=notebook_id, dataset=dataset,\n                                task_id_or_file=function.__name__, pathway="xee",\n                                status="COMPLETED", start_time=start, end_time=utc_now())\n                return result\n            except Exception as exc:\n                logger.error("%s failed: %s\\n%s", function.__name__, exc, traceback.format_exc())\n                append_registry(root, notebook_id=notebook_id, dataset=dataset,\n                                task_id_or_file=function.__name__, pathway="xee",\n                                status="FAILED", start_time=start, end_time=utc_now(),\n                                error_message=str(exc))\n                return None\n        return wrapper\n    return decorator\n\n\nclass BatchManager:\n    """Submit non-blocking Earth Engine exports under a small concurrency cap."""\n\n    def __init__(self, root: Path, notebook_id: str, dataset: str,\n                 logger: logging.Logger, max_concurrent: int = 20,\n                 max_daily_tasks: int = 2500) -> None:\n        self.root = Path(root)\n        self.notebook_id = notebook_id\n        self.dataset = dataset\n        self.logger = logger\n        self.max_concurrent = max_concurrent\n        self.max_daily_tasks = max_daily_tasks\n\n    def active_count(self) -> int:\n        """Count locally registered active submissions."""\n        frame = pd.read_csv(registry_path(self.root))\n        if frame.empty:\n            return 0\n        return int(frame["status"].isin(["STARTED", "RUNNING"]).sum())\n\n    def wait_for_slot(self) -> None:\n        """Wait before submitting when the local active cap is reached."""\n        while self.active_count() >= self.max_concurrent:\n            self.logger.info("Active cap reached; sleeping 60 seconds")\n            time.sleep(60)\n\n    def start(self, task: Any, description: str, retries: int = 3) -> str:\n        """Start a task with retry backoff and return its Earth Engine ID."""\n        self.wait_for_slot()\n        delays = [10, 60, 300]\n        start_time = utc_now()\n        for attempt in range(retries + 1):\n            try:\n                task.start()\n                task_id = getattr(task, "id", "") or task.status().get("id", "")\n                append_registry(self.root, notebook_id=self.notebook_id, dataset=self.dataset,\n                                task_id_or_file=task_id, pathway="batch", status="STARTED",\n                                start_time=start_time, retries=attempt)\n                time.sleep(2)\n                return task_id\n            except Exception as exc:\n                message = str(exc).lower()\n                retryable = "429" in message or "rate limit" in message or "quota" in message\n                if not retryable or attempt >= retries:\n                    append_registry(self.root, notebook_id=self.notebook_id, dataset=self.dataset,\n                                    task_id_or_file=description, pathway="batch", status="FAILED",\n                                    start_time=start_time, end_time=utc_now(),\n                                    error_message=str(exc), retries=attempt)\n                    self.logger.error("Export failed: %s", exc)\n                    return ""\n                delay = delays[min(attempt, len(delays) - 1)]\n                self.logger.warning("Rate limit on %s; retrying in %ss", description, delay)\n                append_registry(self.root, notebook_id=self.notebook_id, dataset=self.dataset,\n                                task_id_or_file=description, pathway="batch", status="RETRIED",\n                                start_time=start_time, retries=attempt + 1,\n                                error_message=str(exc))\n                time.sleep(delay)\n        return ""\n\n    def submit_image(self, image: Any, description: str, folder: str,\n                     prefix: str, region: Any, scale: int, crs: str) -> str:\n        """Create and start a Drive GeoTIFF export without polling."""\n        import ee\n        task = ee.batch.Export.image.toDrive(\n            image=image, description=description, folder=folder,\n            fileNamePrefix=prefix, region=region, scale=scale, crs=crs,\n            maxPixels=1e13, fileFormat="GeoTIFF",\n        )\n        return self.start(task, description)\n\n    def monitor_once(self) -> pd.DataFrame:\n        """Poll registered Earth Engine tasks once and update their statuses."""\n        import ee\n        path = registry_path(self.root)\n        frame = pd.read_csv(path)\n        if frame.empty:\n            return frame\n        for index, row in frame.iterrows():\n            if row["status"] not in ["STARTED", "RUNNING"] or not row["task_id_or_file"]:\n                continue\n            try:\n                status = ee.data.getTaskStatus(str(row["task_id_or_file"]))[0]\n                state = status.get("state", "UNKNOWN")\n                mapped = {"READY": "RUNNING", "RUNNING": "RUNNING",\n                          "COMPLETED": "COMPLETED", "FAILED": "FAILED",\n                          "CANCELLED": "FAILED"}.get(state, state)\n                frame.loc[index, "status"] = mapped\n                frame.loc[index, "end_time"] = utc_now() if mapped in TERMINAL else ""\n                frame.loc[index, "error_message"] = status.get("error_message", "")\n            except Exception as exc:\n                self.logger.warning("Could not poll %s: %s", row["task_id_or_file"], exc)\n        frame.to_csv(path, index=False)\n        return frame\n\n\ndef normalize_projected_dataset(ds: Any, crs: str = "EPSG:5070") -> Any:\n    """Normalize raster dimensions and add projected CF coordinates."""\n    import xarray as xr\n    if "lat" in ds.dims and "y" not in ds.dims:\n        ds = ds.rename({"lat": "y"})\n    if "lon" in ds.dims and "x" not in ds.dims:\n        ds = ds.rename({"lon": "x"})\n    if "y" not in ds.coords:\n        ds = ds.assign_coords(y=np.arange(ds.sizes.get("y", 1), dtype=float))\n    if "x" not in ds.coords:\n        ds = ds.assign_coords(x=np.arange(ds.sizes.get("x", 1), dtype=float))\n    ds["y"].attrs.update({"standard_name": "projection_y_coordinate", "units": "m", "axis": "Y"})\n    ds["x"].attrs.update({"standard_name": "projection_x_coordinate", "units": "m", "axis": "X"})\n    if "lat" not in ds:\n        ds["lat"] = xr.DataArray(np.broadcast_to(ds["y"].values[:, None],\n                                                  (ds.sizes["y"], ds.sizes["x"])), dims=("y", "x"))\n    if "lon" not in ds:\n        ds["lon"] = xr.DataArray(np.broadcast_to(ds["x"].values[None, :],\n                                                  (ds.sizes["y"], ds.sizes["x"])), dims=("y", "x"))\n    ds["lat"].attrs.update({"standard_name": "latitude", "units": "degrees_north"})\n    ds["lon"].attrs.update({"standard_name": "longitude", "units": "degrees_east"})\n    return ds\n\n\ndef write_cf_netcdf(ds: Any, path: Path, attrs: dict[str, Any],\n                    variable_attrs: dict[str, dict[str, Any]] | None = None) -> Path:\n    """Write a compressed projected NetCDF with the Phase 0 CF contract."""\n    import xarray as xr\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    ds = normalize_projected_dataset(ds)\n    ds.attrs.update({\n        "Conventions": "CF-1.8", "title": attrs.get("title", "Ogallala Phase 0 dataset"),\n        "history": f"{utc_now()} notebook={attrs.get(\'notebook_id\', \'unknown\')} git_commit=placeholder",\n        "institution": attrs.get("institution", "Ogallala Phase 0 research team"),\n        "source": attrs.get("source", "Google Earth Engine"),\n        "gee_asset_id": attrs.get("gee_asset_id", ""),\n        "references": attrs.get("references", "Groundwater Buffering of Ecosystem Function During Drought Phase 0 plan"),\n        "comment": attrs.get("comment", "Master grid: EPSG:5070; native bands and QA bands preserved."),\n        "featureType": attrs.get("featureType", "grid"),\n    })\n    ds["crs"] = xr.DataArray(0, attrs={\n        "grid_mapping_name": "albers_conical_equal_area", "semi_major_axis": 6378137.0,\n        "inverse_flattening": 298.257222101, "false_easting": 0.0, "false_northing": 0.0,\n        "spatial_ref": crs_wkt(),\n    })\n    variable_attrs = variable_attrs or {}\n    for name, variable in ds.data_vars.items():\n        if name == "crs":\n            continue\n        defaults = {"long_name": name, "units": "1", "_FillValue": -9999.0,\n                    "grid_mapping": "crs", "coordinates": "lat lon"}\n        defaults.update(variable_attrs.get(name, {}))\n        variable.attrs.update(defaults)\n    encoding: dict[str, dict[str, Any]] = {}\n    for name, variable in ds.data_vars.items():\n        if name == "crs":\n            continue\n        fill = variable.attrs.get("_FillValue", -9999.0)\n        encoding[name] = {"zlib": True, "complevel": 4, "_FillValue": fill}\n        if "time" in variable.dims:\n            encoding[name]["chunksizes"] = tuple(min(24, ds.sizes[d]) for d in variable.dims)\n    if "time" in ds.coords:\n        ds["time"].attrs.update({"standard_name": "time", "long_name": "time",\n                                  "units": "days since 1970-01-01 00:00:00",\n                                  "calendar": "proleptic_gregorian", "axis": "T"})\n        encoding["time"] = {"units": "days since 1970-01-01 00:00:00",\n                             "calendar": "proleptic_gregorian"}\n    ds.to_netcdf(path, format="NETCDF4_CLASSIC", engine="netcdf4", encoding=encoding)\n    return path\n\n\ndef crs_wkt() -> str:\n    """Return a WKT representation for the master Albers grid."""\n    try:\n        from pyproj import CRS\n        return CRS.from_epsg(5070).to_wkt()\n    except Exception:\n        return "EPSG:5070"\n\n\ndef qc_netcdf(path: Path, qc_dir: Path) -> dict[str, Any]:\n    """Create summary statistics and a quicklook for a NetCDF file."""\n    import matplotlib.pyplot as plt\n    import xarray as xr\n    path, qc_dir = Path(path), Path(qc_dir)\n    qc_dir.mkdir(parents=True, exist_ok=True)\n    ds = xr.open_dataset(path)\n    summary: dict[str, Any] = {}\n    for name, value in ds.data_vars.items():\n        if name == "crs":\n            continue\n        array = value.values.astype(float)\n        finite = array[np.isfinite(array)]\n        summary[name] = {"min": float(np.min(finite)) if finite.size else None,\n                         "max": float(np.max(finite)) if finite.size else None,\n                         "mean": float(np.mean(finite)) if finite.size else None,\n                         "std": float(np.std(finite)) if finite.size else None,\n                         "n_valid": int(finite.size)}\n        if finite.size:\n            image = np.nanmean(array, axis=0) if "time" in value.dims else array\n            plt.imsave(qc_dir / f"{path.stem}_{name}.png", image, cmap="viridis")\n    summary_path = qc_dir / f"{path.stem}_summary_stats.json"\n    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    ds.close()\n    return summary\n\n\ndef write_manifest(root: Path, notebook_id: str, files: Iterable[Path],\n                   dataset: str, status: str = "COMPLETED") -> Path:\n    """Write provenance metadata for files produced by one notebook."""\n    entries = []\n    for file_path in files:\n        file_path = Path(file_path)\n        if file_path.exists():\n            entries.append({"file": str(file_path), "size_bytes": file_path.stat().st_size,\n                            "cf_version": "CF-1.8", "dataset": dataset})\n    path = Path(root) / "manifests" / f"{notebook_id}_manifest.json"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps({"notebook_id": notebook_id, "status": status,\n                                "files": entries}, indent=2), encoding="utf-8")\n    return path\n'
(SETUP_DIR / 'utils.py').write_text(UTILS_SOURCE, encoding='utf-8')
sys.path.insert(0, str(SETUP_DIR))
from utils import *
NOTEBOOK_ID = '99_run_all_and_validate'


In [ ]:
import yaml
CONFIG = yaml.safe_load((DRIVE_ROOT / '00_setup/config.yaml').read_text())
DRY_RUN = CONFIG.get('DRY_RUN', False)


In [ ]:
logger, LOG_PATH = configure_logger(NOTEBOOK_ID, DRIVE_ROOT)


In [ ]:
DATASET = {'name': 'All Phase 0 notebooks', 'drive_subfolder': ''}


In [ ]:
# Notebook execution is controlled by RUN_NOTEBOOKS=1 to keep normal validation non-blocking.


In [ ]:
# Each dataset notebook owns its XEE or batch pathway.


In [ ]:
# Each dataset notebook owns its Drive exports.


In [ ]:
MAX_CONCURRENT = 20
MAX_DAILY_TASKS = 2500


In [ ]:
# CF validation is performed in the aggregation cell below.


In [ ]:
# QC and manifest aggregation are included in the README generation below.


In [ ]:
import subprocess
import sys
import pandas as pd
import xarray as xr

NOTEBOOK_DIR = Path.cwd()
# Upload the generated notebooks to this Drive folder, or set PHASE0_NOTEBOOK_DIR.
NOTEBOOK_DIR = Path(os.getenv('PHASE0_NOTEBOOK_DIR', str(NOTEBOOK_DIR)))
notebooks = sorted(NOTEBOOK_DIR.glob('00_*.ipynb')) + sorted(NOTEBOOK_DIR.glob('01*.ipynb')) + sorted(NOTEBOOK_DIR.glob('02*.ipynb'))
if os.getenv('RUN_NOTEBOOKS', '0') == '1':
    for notebook_path in notebooks:
        subprocess.run([sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', str(notebook_path), '--output', str(notebook_path)], check=True)

failures = []
cf_files = list(DRIVE_ROOT.glob('01_gee_rasters/**/cf/*.nc')) + list(DRIVE_ROOT.glob('02_independent/**/cf/*.nc'))
for path in cf_files:
    try:
        with xr.open_dataset(path) as ds:
            required = {'Conventions', 'title', 'history', 'institution', 'source', 'references', 'comment'}
            missing = required - set(ds.attrs)
            if ds.attrs.get('Conventions') != 'CF-1.8' or missing:
                failures.append(f'{path}: missing CF attrs {sorted(missing)}')
    except Exception as exc:
        failures.append(f'{path}: {exc}')
registry = pd.read_csv(registry_path(DRIVE_ROOT))
if not registry.empty:
    failures.extend(registry.loc[registry.status == 'FAILED', 'error_message'].dropna().astype(str).tolist())
print(registry.groupby('status').size().to_string() if not registry.empty else 'No registry rows')
readme = DRIVE_ROOT / 'README.md'
lines = ['# Ogallala Phase 0 output index', '', f'Generated: {utc_now()}', '', '## CF files']
lines.extend(f'- `{path}` ({path.stat().st_size} bytes)' for path in cf_files)
lines += ['', '## Validation', f'- CF files checked: {len(cf_files)}', f'- Failures: {len(failures)}']
readme.write_text('\n'.join(lines) + '\n', encoding='utf-8')
if failures:
    raise RuntimeError('Phase 0 validation failures:\n' + '\n'.join(failures))
print('Validation passed; README written to', readme)
